In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [2]:
spark = (
    SparkSession.builder
    .appName("aula_08_parquet")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/30 18:33:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df_csv = spark.read.csv("/workspace/datasets/08_parquet/pedidos.csv", header= True, inferSchema= True)

In [4]:
df_csv.printSchema()

root
 |-- pedido_id: integer (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- data_pedido: date (nullable = true)
 |-- status: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- valor_total: double (nullable = true)



In [5]:
df_csv.show(5)

+---------+----------+-----------+---------+-----------+-----------+
|pedido_id|cliente_id|data_pedido|   status|      canal|valor_total|
+---------+----------+-----------+---------+-----------+-----------+
|    10001|       452| 2025-10-09|concluido|       site|     473.56|
|    10002|        71| 2025-08-03|concluido|        app|     303.13|
|    10003|       411| 2025-10-06|  enviado|       loja|     281.64|
|    10004|       184| 2025-09-03|concluido|       loja|     281.65|
|    10005|       290| 2025-09-15|concluido|marketplace|     893.22|
+---------+----------+-----------+---------+-----------+-----------+
only showing top 5 rows



In [6]:
df_csv.write.mode("overwrite").parquet("/workspace/datasets/08_parquet/arquivos_parquet")

In [7]:
df_parquet = spark.read.parquet("/workspace/datasets/08_parquet/arquivos_parquet")

In [8]:
df_parquet.printSchema()

root
 |-- pedido_id: integer (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- data_pedido: date (nullable = true)
 |-- status: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- valor_total: double (nullable = true)



In [9]:
df_parquet_select = (
df_parquet.select("cliente_id", "valor_total")
)

In [10]:
df_parquet_select.explain(True)

== Parsed Logical Plan ==
'Project ['cliente_id, 'valor_total]
+- Relation [pedido_id#73,cliente_id#74,data_pedido#75,status#76,canal#77,valor_total#78] parquet

== Analyzed Logical Plan ==
cliente_id: int, valor_total: double
Project [cliente_id#74, valor_total#78]
+- Relation [pedido_id#73,cliente_id#74,data_pedido#75,status#76,canal#77,valor_total#78] parquet

== Optimized Logical Plan ==
Project [cliente_id#74, valor_total#78]
+- Relation [pedido_id#73,cliente_id#74,data_pedido#75,status#76,canal#77,valor_total#78] parquet

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [cliente_id#74,valor_total#78] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/workspace/datasets/08_parquet/arquivos_parquet], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<cliente_id:int,valor_total:double>



In [11]:
df_pushdown = (
    df_parquet.select("cliente_id", "valor_total")
    .filter(F.col("valor_total") > 500)
)

In [12]:
df_pushdown.explain(True)

== Parsed Logical Plan ==
'Filter ('valor_total > 500)
+- Project [cliente_id#74, valor_total#78]
   +- Relation [pedido_id#73,cliente_id#74,data_pedido#75,status#76,canal#77,valor_total#78] parquet

== Analyzed Logical Plan ==
cliente_id: int, valor_total: double
Filter (valor_total#78 > cast(500 as double))
+- Project [cliente_id#74, valor_total#78]
   +- Relation [pedido_id#73,cliente_id#74,data_pedido#75,status#76,canal#77,valor_total#78] parquet

== Optimized Logical Plan ==
Project [cliente_id#74, valor_total#78]
+- Filter (isnotnull(valor_total#78) AND (valor_total#78 > 500.0))
   +- Relation [pedido_id#73,cliente_id#74,data_pedido#75,status#76,canal#77,valor_total#78] parquet

== Physical Plan ==
*(1) Filter (isnotnull(valor_total#78) AND (valor_total#78 > 500.0))
+- *(1) ColumnarToRow
   +- FileScan parquet [cliente_id#74,valor_total#78] Batched: true, DataFilters: [isnotnull(valor_total#78), (valor_total#78 > 500.0)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file

In [13]:
df_csv.write.mode("overwrite").parquet("/workspace/datasets/08_parquet/pedidos_normal")

In [14]:
df_parquet_pedidos = spark.read.parquet("/workspace/datasets/08_parquet/pedidos_normal")

In [15]:
df_csv.write.mode("overwrite").partitionBy("status").parquet("/workspace/datasets/08_parquet/pedidos_por_status")

In [16]:
df_pedidos_status = spark.read.parquet("/workspace/datasets/08_parquet/pedidos_por_status")

In [17]:
df_pedidos_status.explain(True)

== Parsed Logical Plan ==
Relation [pedido_id#127,cliente_id#128,data_pedido#129,canal#130,valor_total#131,status#132] parquet

== Analyzed Logical Plan ==
pedido_id: int, cliente_id: int, data_pedido: date, canal: string, valor_total: double, status: string
Relation [pedido_id#127,cliente_id#128,data_pedido#129,canal#130,valor_total#131,status#132] parquet

== Optimized Logical Plan ==
Relation [pedido_id#127,cliente_id#128,data_pedido#129,canal#130,valor_total#131,status#132] parquet

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [pedido_id#127,cliente_id#128,data_pedido#129,canal#130,valor_total#131,status#132] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/workspace/datasets/08_parquet/pedidos_por_status], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<pedido_id:int,cliente_id:int,data_pedido:date,canal:string,valor_total:double>



In [18]:
df_status_pruning = (
    df_pedidos_status.filter(F.col("status") == "concluido")
)

In [19]:
df_status_pruning.explain(True)

== Parsed Logical Plan ==
'Filter ('status = concluido)
+- Relation [pedido_id#127,cliente_id#128,data_pedido#129,canal#130,valor_total#131,status#132] parquet

== Analyzed Logical Plan ==
pedido_id: int, cliente_id: int, data_pedido: date, canal: string, valor_total: double, status: string
Filter (status#132 = concluido)
+- Relation [pedido_id#127,cliente_id#128,data_pedido#129,canal#130,valor_total#131,status#132] parquet

== Optimized Logical Plan ==
Filter (isnotnull(status#132) AND (status#132 = concluido))
+- Relation [pedido_id#127,cliente_id#128,data_pedido#129,canal#130,valor_total#131,status#132] parquet

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [pedido_id#127,cliente_id#128,data_pedido#129,canal#130,valor_total#131,status#132] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/workspace/datasets/08_parquet/pedidos_por_status], PartitionFilters: [isnotnull(status#132), (status#132 = concluido)], PushedFilters: [], Rea

In [20]:
#cardinalidade gigantesca decisão ruim salvar por partição
#df_csv.write.mode("overwrite").partitionBy("cliente_id").parquet("/workspace/datasets/08_parquet/pedidos_por_clientes")

In [ ]:
spark.stop()

26/09/30 00:55:48 ERROR FileFormatWriter: Aborting job d328cccc-af2c-47f8-abac-e35e33fc9eac.
java.util.NoSuchElementException: None.get
	at scala.None$.get(Option.scala:529)
	at scala.None$.get(Option.scala:527)
	at org.apache.spark.sql.execution.datasources.BasicWriteJobStatsTracker.processStats(BasicWriteStatsTracker.scala:202)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.$anonfun$processStats$5(FileFormatWriter.scala:448)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.$anonfun$processStats$5$adapted(FileFormatWriter.scala:447)
	at scala.collection.immutable.List.foreach(List.scala:431)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.processStats(FileFormatWriter.scala:447)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.writeAndCommit(FileFormatWriter.scala:278)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.executeWrite(FileFormatWriter.scala:304)
	at org.apache.spark.sql.execution.datasources